In [1]:
from pathlib import Path
import scanpy as sc
import scgpt as scg


c:\Users\acer\anaconda3\envs\sc-embedding-benchmark\lib\site-packages\scgpt\model\model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
c:\Users\acer\anaconda3\envs\sc-embedding-benchmark\lib\site-packages\scgpt\model\multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
c:\Users\acer\anaconda3\envs\sc-embedding-benchmark\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
adata = sc.read_h5ad("data/processed/pbmc3k_annotated_benchmark.h5ad")


In [3]:
model_dir = Path("weights/scgpt")



In [8]:
import os

if not hasattr(os, "sched_getaffinity"):
    os.sched_getaffinity = lambda pid: set(range(os.cpu_count() or 1))

In [13]:
import os
import scgpt.tasks.cell_emb as cell_emb

# Force scGPT DataLoader to use num_workers=0 on Windows
os.sched_getaffinity = lambda pid: set()
cell_emb.os.sched_getaffinity = lambda pid: set()

print("Forced worker count:", len(cell_emb.os.sched_getaffinity(0)))

embedded_adata = scg.tasks.embed_data(
    adata,
    model_dir,
    gene_col="gene_symbols",
    batch_size=64,
    device="cpu",
    use_fast_transformer=False,
    return_new_adata=True,
)

Forced worker count: 0
scGPT - INFO - match 1949/2000 genes in vocabulary of size 60697.


Embedding cells:   0%|          | 0/184 [00:00<?, ?it/s]c:\Users\acer\anaconda3\envs\sc-embedding-benchmark\lib\site-packages\torch\nn\modules\transformer.py:380: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ..\aten\src\ATen\NestedTensorImpl.cpp:180.)
  output = torch._nested_tensor_from_mask(output, src_key_padding_mask.logical_not(), mask_check=False)
Embedding cells: 100%|██████████| 184/184 [37:12<00:00, 12.13s/it]


In [15]:
adata.obsm["X_scgpt"] = embedded_adata.X.copy()


In [16]:
adata.write("data/processed/pbmc_scgpt_embedded.h5ad")

In [17]:
adata = sc.read_h5ad("data/processed/pbmc_scgpt_embedded.h5ad")

In [18]:
"X_scgpt" in adata.obsm

True

In [19]:
import numpy as np
import os

os.makedirs("results/embeddings", exist_ok=True)

np.save(
    "results/embeddings/X_scgpt.npy",
    adata.obsm["X_scgpt"]
)

In [20]:
adata.obsm["X_scgpt"] = np.load(
    "results/embeddings/X_scgpt.npy"
)